# 03 - Evaluasi model

Notebook ini dijalankan setelah training selesai dan `best.pt` tersedia. Evaluasi utama memakai `test`, lalu evaluasi tambahan memakai `external_holdout`. External tidak pernah dipakai untuk training.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

PACKAGE = Path.cwd()
PACKAGE = next((p for p in (PACKAGE, *PACKAGE.parents) if (p / 'dataset' / 'data.yaml').is_file()), None) or next((p.parent.parent for p in Path.cwd().glob('**/dataset/data.yaml')), None)
if PACKAGE is None:
    raise FileNotFoundError('Folder paket tomato-pumpkin-lemon-training belum dibuka.')

RUN_NAME = 'adaptive-10ep'  # ubah sesuai nama run yang benar-benar selesai
RUN_DIR = PACKAGE / 'runs' / 'adaptive' / RUN_NAME
BEST = RUN_DIR / 'weights' / 'best.pt'
EVALUATOR = PACKAGE / 'training' / 'evaluate_tomato_pumpkin_lemon.py'
DATA_YAML = PACKAGE / 'dataset' / 'data.yaml'
EXTERNAL_YAML = PACKAGE / 'dataset' / 'data-external.yaml'
MANIFEST = PACKAGE / 'dataset' / 'split_manifest.json'
OUTPUT = RUN_DIR / 'notebook-evaluation.json'

if not BEST.is_file():
    raise FileNotFoundError(f'best.pt belum ada: {BEST}. Jalankan notebook 02 terlebih dahulu.')
print('Checkpoint:', BEST)


In [ ]:
command = [
    sys.executable, str(EVALUATOR),
    '--weights', str(BEST),
    '--data', str(DATA_YAML),
    '--external-data', str(EXTERNAL_YAML),
    '--manifest', str(MANIFEST),
    '--output', str(OUTPUT),
    '--device', 'auto',
]
print('Test utama dan external holdout dijalankan setelah checkpoint tersedia.')
result = subprocess.run(command, cwd=PACKAGE)
if result.returncode not in (0, 2):
    raise RuntimeError(f'Evaluasi gagal dengan exit code {result.returncode}.')
print('Laporan:', OUTPUT)


In [ ]:
payload = json.loads(OUTPUT.read_text(encoding='utf-8'))
gate = payload['gate']
print('Status gate:', gate['status'])
for name, values in payload['metrics'].get('per_class', {}).items():
    print(f"{name:10} precision={values['precision']:.4f} recall={values['recall']:.4f} mAP50={values['map50']:.4f} mAP50-95={values['map50_95']:.4f} support={values['support']}")
if 'external' in payload['metrics']:
    print('External holdout tersedia di payload[\"metrics\"][\"external\"].')
